# Week 11: Advanced Specialized Machine Learning
## Notebook 10: Conformal Prediction: Distribution-Free Uncertainty Guarantees

### 1. Conceptual Intuition & Theoretical Background
Standard ML models output point predictions or overconfident softmax probabilities with zero formal guarantees of correctness.

**Conformal Prediction (Vovk et al., 2005):**
Transforms any black-box machine learning model into a **statistically rigorous prediction set** $C(X) \subseteq \mathcal{Y}$ satisfying exact finite-sample coverage:
$$P(Y \in C(X_{\text{test}})) \ge 1 - \alpha$$
Where $1 - \alpha$ is the user-specified confidence level (e.g., $95\%$), requiring **only exchangeability** (weaker than i.i.d.).

**Split Conformal Algorithm:**
1. Split calibration dataset $(X_{\text{cal}}, y_{\text{cal}})$ of size $n$.
2. Compute non-conformity score $s_i = s(X_i, y_i)$ (e.g. $s_i = 1 - \hat{P}(y_i \mid X_i)$).
3. Compute the adjusted quantile threshold:
$$\hat{q} = \text{Quantile}\left( s_1, \dots, s_n;\ \frac{\lceil (n + 1)(1 - \alpha) \rceil}{n} \right)$$
4. For test point $X_{\text{test}}$, prediction set includes all classes whose score falls below $\hat{q}$:
$$C(X_{\text{test}}) = \{ y \in \mathcal{Y} : s(X_{\text{test}}, y) \le \hat{q} \}$$

---

### 2. Scratch Python Implementation: Conformal Classification Sets

In [ ]:
import numpy as np

class ConformalClassifierScratch:
    def __init__(self, alpha=0.10):
        self.alpha = alpha
        self.q_hat = None
        
    def calibrate(self, cal_probabilities, cal_true_labels):
        """
        cal_probabilities: shape (N_cal, K_classes)
        cal_true_labels: ground truth 1D array
        """
        n_cal = len(cal_true_labels)
        # Non-conformity score: s_i = 1 - P_hat(true_class)
        true_class_probs = cal_probabilities[np.arange(n_cal), cal_true_labels]
        scores = 1.0 - true_class_probs
        
        # Conformal quantile level: ceil((n + 1) * (1 - alpha)) / n
        quantile_level = np.ceil((n_cal + 1) * (1.0 - self.alpha)) / n_cal
        quantile_level = min(1.0, quantile_level)
        
        self.q_hat = np.quantile(scores, quantile_level)
        return self.q_hat
        
    def predict_sets(self, test_probabilities):
        """Returns prediction sets: all classes with 1 - P(c) <= q_hat"""
        # Equivalent to P(c) >= 1 - q_hat
        threshold = 1.0 - self.q_hat
        prediction_sets = []
        for probs in test_probabilities:
            candidate_classes = np.where(probs >= threshold)[0]
            prediction_sets.append(candidate_classes.tolist())
        return prediction_sets

# Test on 3-Class Synthetic Model Softmax Predictions
np.random.seed(42)
n_cal = 500
n_test = 200
n_classes = 3

# Synthetic calibration data
cal_probs = np.random.dirichlet([2, 1, 1], size=n_cal)
cal_labels = np.array([np.random.choice(n_classes, p=p) for p in cal_probs])

# Synthetic test data
test_probs = np.random.dirichlet([2, 1, 1], size=n_test)
test_labels = np.array([np.random.choice(n_classes, p=p) for p in test_probs])

# Calibrate for 90% target coverage (alpha = 0.10)
conformal = ConformalClassifierScratch(alpha=0.10)
q_threshold = conformal.calibrate(cal_probs, cal_labels)

pred_sets = conformal.predict_sets(test_probs)

# Empirical Coverage on Test Data
empirical_coverage = np.mean([
    test_labels[i] in pred_sets[i] for i in range(n_test)
]) * 100

mean_set_size = np.mean([len(s) for s in pred_sets])

print("--- Split Conformal Classification ---")
print(f"Target Confidence Level (1 - alpha): 90.0%")
print(f"Fitted Conformal Threshold (q_hat):  {q_threshold:.4f}")
print(f"Empirical Test Coverage Observed:    {empirical_coverage:.1f}% (Guaranteed >= 90%!)")
print(f"Average Prediction Set Cardinality:  {mean_set_size:.2f} classes per sample")

### 3. Step-by-Step Code Walkthrough
1. **Non-Conformity Scoring:** Scores how unusual the true label is according to model probabilities.
2. **Quantile Correction:** Adjusts the quantile to guarantee finite-sample coverage without asymptotic approximations.
3. **Adaptive Set Cardinality:** Easy examples produce singleton sets (e.g. `[0]`), while ambiguous examples expand into multi-class sets (e.g. `[0, 2]`).

---

### 4. Industry Applications & Trade-offs
- **Medical Diagnostics & Autonomous Driving:** Handing over cases to human specialists whenever prediction set size exceeds 1.
- **Trade-off:** Guarantees marginal coverage across the population, not conditional coverage for every specific sub-group.

---

### 5. Top Interview Questions & Answers
**Q1: Does Conformal Prediction require retraining the underlying machine learning model?**
> **Answer:** No. Conformal prediction is completely model-agnostic. It works as a wrapper around any pre-trained black-box model (neural network, gradient boosted trees, LLM) using a calibration split.

**Q2: What is the only core assumption required for Conformal Prediction coverage guarantees to hold?**
> **Answer:** Exchangeability of calibration and test data points (weaker than independent and identically distributed, i.i.d.). It does not require any parametric distribution assumptions.